In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 17.7: MNIST reconstructions using linear and nonlinear fully-connected autoencoders

## What is an Autoencoder?

An **autoencoder** is a neural network trained to reconstruct its input through a compressed bottleneck representation called the **latent code**:

$$z = f_{\text{enc}}(x) \in \mathbb{R}^k, \quad \hat{x} = f_{\text{dec}}(z) \approx x$$

where $k \ll d$ (latent dim much smaller than input dim $d$). Training minimises **reconstruction loss**:

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} \|x_i - \hat{x}_i\|^2$$

---

## Linear AE = PCA

A **linear autoencoder** uses no nonlinear activations:

$$z = W_{\text{enc}}\, x \in \mathbb{R}^k, \qquad \hat{x} = W_{\text{dec}}\, z$$

The loss is:

$$\mathcal{L} = \|x - W_{\text{dec}}\,W_{\text{enc}}\, x\|^2$$

**Key result:** For a one-hidden-layer linear AE trained with MSE loss and *no* bias terms, the columns of $W_{\text{dec}}$ span the same subspace as the **top-$k$ principal components** of the data covariance matrix. This is the exact PCA subspace — the AE and PCA give identical reconstructions (up to an orthogonal rotation of the latent space).

---

## Nonlinear AE

Adding nonlinear activations $\phi$ breaks the PCA equivalence and allows the encoder to learn **nonlinear manifolds**:

$$z = \phi(W_{\text{enc}}\, x), \qquad \hat{x} = \sigma(W_{\text{dec}}\, z)$$

Common choices: `sigmoid`, `tanh`, `relu`, `leaky_relu`. Nonlinear AEs can achieve lower reconstruction error for the same $k$ on complex data like images.

---

**This notebook** trains a simple single-hidden-layer AE on the sklearn digits dataset (8×8 = 64 features, 10 classes) using pure numpy gradient descent and visualises:
1. Training loss curve
2. Sample reconstructions
3. 2D latent space (PCA of latent representations)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from sklearn import datasets

digits = datasets.load_digits()
X = digits.data / 16.0  # normalise to [0, 1], shape (1797, 64)
y = digits.target

# ── Activation functions and derivatives ──────────────────────────────────────
def sigmoid(x):      return 1 / (1 + np.exp(-np.clip(x, -20, 20)))
def tanh(x):         return np.tanh(x)
def relu(x):         return np.maximum(0, x)
def leakyrelu(x):    return np.where(x > 0, x, 0.01 * x)
def linear(x):       return x

def sigmoid_d(x):    s = sigmoid(x); return s * (1 - s)
def tanh_d(x):       return 1 - np.tanh(x) ** 2
def relu_d(x):       return (x > 0).astype(float)
def leakyrelu_d(x):  return np.where(x > 0, 1.0, 0.01)
def linear_d(x):     return np.ones_like(x)

ACT = {
    'linear':    (linear,    linear_d),
    'sigmoid':   (sigmoid,   sigmoid_d),
    'tanh':      (tanh,      tanh_d),
    'relu':      (relu,      relu_d),
    'leakyrelu': (leakyrelu, leakyrelu_d),
}


# ── Autoencoder training (numpy gradient descent) ─────────────────────────────
def train_ae(act_name='tanh', latent_dim=8, n_epochs=200, lr=0.01, seed=42):
    rng = np.random.default_rng(seed)
    act, act_d = ACT[act_name]
    n_in = 64

    # Xavier-style initialisation
    W_enc = rng.normal(0, np.sqrt(2 / (n_in + latent_dim)), (n_in, latent_dim))
    W_dec = rng.normal(0, np.sqrt(2 / (latent_dim + n_in)), (latent_dim, n_in))

    losses = []
    for epoch in range(n_epochs):
        # ── Forward pass ──
        z_pre = X @ W_enc                  # (N, k)
        z     = act(z_pre)                 # (N, k)
        x_hat_pre = z @ W_dec              # (N, 64)
        x_hat = sigmoid(x_hat_pre)         # sigmoid output for [0,1]

        loss = np.mean((X - x_hat) ** 2)
        losses.append(loss)

        # ── Backward pass ──
        dL_dxhat     = -2 * (X - x_hat) / X.shape[0]
        dL_dxhat_pre = dL_dxhat * sigmoid_d(x_hat_pre)
        dL_dWdec     = z.T @ dL_dxhat_pre
        dL_dz        = dL_dxhat_pre @ W_dec.T
        dL_dzpre     = dL_dz * act_d(z_pre)
        dL_dWenc     = X.T @ dL_dzpre

        W_dec -= lr * dL_dWdec
        W_enc -= lr * dL_dWenc

    # Final representations
    z_final = act(X @ W_enc)
    x_recon = sigmoid(z_final @ W_dec)
    return losses, z_final, x_recon


# ── Visualisation ─────────────────────────────────────────────────────────────
def draw_ae(act_name='tanh', latent_dim=8, n_epochs=200):
    losses, z_final, x_recon = train_ae(act_name, latent_dim, n_epochs)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. Training loss curve (log scale)
    axes[0].semilogy(losses, color='#2563eb', lw=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE (log scale)')
    axes[0].set_title(f'Training loss  |  activation = {act_name}', fontsize=11)
    axes[0].grid(True, which='both', alpha=0.3)

    # 2. Sample reconstructions (first 8 digits, original vs reconstructed)
    n_show = 8
    # Stack original and reconstructed; each image is 8 pixels wide
    # Build a strip: n_show columns, each column = [orig(8px) | recon(8px)]
    strip = np.hstack([
        np.vstack([X[i].reshape(8, 8), x_recon[i].reshape(8, 8)])
        for i in range(n_show)
    ])
    axes[1].imshow(strip, cmap='gray_r', aspect='auto', vmin=0, vmax=1)
    axes[1].set_yticks([])
    axes[1].set_xticks(np.arange(n_show) * 8 + 4)
    axes[1].set_xticklabels([str(digits.target[i]) for i in range(n_show)], fontsize=10)
    axes[1].axhline(7.5, color='#dc2626', lw=1.5, linestyle='--')
    axes[1].set_title(
        f'Top: original  |  Bottom: reconstruction\n(latent_dim = {latent_dim})',
        fontsize=10
    )

    # 3. 2D latent space (PCA of latent representations)
    if latent_dim >= 2:
        z_c = z_final - z_final.mean(0)
        cov = z_c.T @ z_c / len(z_c)
        vals, vecs = np.linalg.eigh(cov)
        idx = np.argsort(vals)[::-1]
        z_2d = z_c @ vecs[:, idx[:2]]
    else:
        z_2d = np.column_stack([z_final[:, 0], np.zeros(len(z_final))])

    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    for digit in range(10):
        mask = y == digit
        axes[2].scatter(
            z_2d[mask, 0], z_2d[mask, 1],
            c=[colors[digit]], s=8, alpha=0.65, label=str(digit)
        )
    axes[2].set_title('2D latent space (top-2 PCs of latent)', fontsize=10)
    axes[2].set_xlabel('PC 1')
    axes[2].set_ylabel('PC 2')
    axes[2].legend(fontsize=7, ncol=5, loc='upper right', markerscale=2)

    plt.suptitle(
        f'{act_name} AE  |  latent_dim={latent_dim}  |  final MSE={losses[-1]:.4f}',
        fontsize=10, y=0.0
    )
    plt.tight_layout()
    plt.show()


# ── Interactive widget ────────────────────────────────────────────────────────
widgets.interact(
    draw_ae,
    act_name=widgets.Dropdown(
        options=['linear', 'sigmoid', 'tanh', 'relu', 'leakyrelu'],
        value='tanh',
        description='Activation',
        style={'description_width': 'initial'}
    ),
    latent_dim=widgets.IntSlider(
        min=1, max=32, step=1, value=8,
        description='Latent dim',
        continuous_update=False,
        style={'description_width': 'initial'}
    ),
    n_epochs=widgets.IntSlider(
        min=50, max=500, step=50, value=200,
        description='Epochs',
        continuous_update=False,
        style={'description_width': 'initial'}
    ),
)